In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import kagglehub
# Download latest version
path = kagglehub.competition_download('iitb-itc-aiml-team-selection')
print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/iitb-itc-aiml-team-selection


In [2]:
import os
import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import xgboost as xgb
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# New imports for enhanced features
from skimage.feature import hog, local_binary_pattern
from skimage.filters import gabor
from skimage import exposure

warnings.filterwarnings('ignore')
np.random.seed(37)

print("All imports successful!")

All imports successful!


In [3]:
# Load the CSV
df_label = pd.read_csv("/kaggle/input/competitions/iitb-itc-aiml-team-selection/train.csv")

# Create label mapping (0-6)
unique_labels = sorted(df_label['label'].unique())
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
inverse_mapping = {idx: label for label, idx in label_mapping.items()}

print(f"Classes: {unique_labels}")
print(f"Label mapping: {label_mapping}")
print(f"Total samples: {len(df_label)}")

Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Label mapping: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
Total samples: 28709


In [4]:
# Split data (85/15 with stratification)
train_df, val_df = train_test_split(
    df_label, 
    test_size=0.15, 
    random_state=37, 
    stratify=df_label['label']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print("\nTraining label distribution:")
print(train_df['label'].value_counts(normalize=True))

Training samples: 24402
Validation samples: 4307

Training label distribution:
label
happy       0.251332
neutral     0.172937
sad         0.168224
fear        0.142693
angry       0.139169
surprise    0.110442
disgust     0.015204
Name: proportion, dtype: float64


In [5]:
def extract_enhanced_features(image_path):
    """
    Extract:
    1. HOG (900 features)
    2. Multi-scale LBP (R=1,2,3) (~59 + 243 + 243 = 545 features)
    3. Geometric ratios (approx regions) (12 features)
    4. Gabor filters (4 orientations x 3 scales x 2 stats = 24 features)
    Total: ~1481 features
    """
    # Read image
    image = cv2.imread(image_path)
    if image is None:
        return None
    
    # Grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    
    # Resize to 48x48
    if gray.shape != (48, 48):
        gray = cv2.resize(gray, (48, 48))
    
    features = []
    
    # ----- 1. HOG (900 features) -----
    hog_feats = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=False,
        block_norm='L2-Hys',
        transform_sqrt=True
    )
    features.extend(hog_feats)
    
    # ----- 2. Multi-scale LBP (R=1, R=2, R=3) -----
    def get_lbp_hist(image, radius, n_points=None):
        if n_points is None:
            n_points = 8 * radius
        lbp = local_binary_pattern(image, P=n_points, R=radius, method='uniform')
        # Uniform LBP returns 0..P+1, so bins = P+2
        n_bins = n_points + 2
        hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
        hist = hist.astype(float) / (48 * 48)
        return hist
    
    # R=1 (8 neighbors, 59 bins)
    lbp_r1 = get_lbp_hist(gray, radius=1, n_points=8)
    features.extend(lbp_r1)
    
    # R=2 (16 neighbors, ~243 bins)
    lbp_r2 = get_lbp_hist(gray, radius=2, n_points=16)
    features.extend(lbp_r2)
    
    # R=3 (24 neighbors, ~243 bins - but capped to keep size manageable)
    lbp_r3 = get_lbp_hist(gray, radius=3, n_points=24)
    features.extend(lbp_r3)
    
    # ----- 3. Geometric / Region-based features (approximated) -----
    h, w = gray.shape
    # Define fixed facial regions on 48x48 (assuming centered face)
    # Left eye: rows 10-22, cols 8-20
    left_eye = gray[10:22, 8:20]
    # Right eye: rows 10-22, cols 28-40
    right_eye = gray[10:22, 28:40]
    # Mouth: rows 28-38, cols 14-34
    mouth = gray[28:38, 14:34]
    # Brow region: rows 4-12, cols 10-38
    brow = gray[4:12, 10:38]
    
    # Statistics
    features.extend([
        np.mean(left_eye), np.std(left_eye),
        np.mean(right_eye), np.std(right_eye),
        np.mean(mouth), np.std(mouth),
        np.mean(brow), np.std(brow)
    ])
    
    # Ratios (expression indicators)
    eye_avg = (np.mean(left_eye) + np.mean(right_eye)) / 2
    mouth_mean = np.mean(mouth)
    features.append(eye_avg / (mouth_mean + 0.001))  # Eye-to-mouth brightness (squinting vs open)
    features.append(np.std(left_eye) / (np.std(right_eye) + 0.001))  # Eye asymmetry (disgust)
    features.append(np.mean(brow) / (eye_avg + 0.001))  # Brow-to-eye ratio (furrowed vs raised)
    features.append(np.std(mouth) / (np.mean(mouth) + 0.001))  # Mouth texture variation (tension)
    
    # ----- 4. Gabor filters (4 orientations x 3 scales) -----
    gabor_features = []
    for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:  # 0, 45, 90, 135 deg
        for frequency in [0.1, 0.25, 0.4]:           # low, mid, high frequency
            real, imag = gabor(gray, frequency=frequency, theta=theta)
            magnitude = np.sqrt(real**2 + imag**2)
            gabor_features.extend([np.mean(magnitude), np.std(magnitude)])
    features.extend(gabor_features)
    
    # Convert to numpy array and handle any NaN
    features = np.array(features)
    if np.isnan(features).any():
        features = np.nan_to_num(features, nan=0.0)
    
    return features

# Test on a sample to show feature size
img_dir = "/kaggle/input/competitions/iitb-itc-aiml-team-selection/train/"
sample_path = os.path.join(img_dir, pd.read_csv("/kaggle/input/competitions/iitb-itc-aiml-team-selection/train.csv").iloc[0]['id'])
sample_feats = extract_enhanced_features(sample_path)
print(f"✅ Enhanced feature vector size: {len(sample_feats)}")

✅ Enhanced feature vector size: 990


In [6]:
img_dir = "/kaggle/input/competitions/iitb-itc-aiml-team-selection/train/"

train_features = []
train_labels = []
train_failed = []

print("Extracting training features (HOG+LBP)...")
for idx, row in train_df.iterrows():
    img_path = os.path.join(img_dir, row['id'])
    feats = extract_enhanced_features(img_path)
    if feats is not None:
        train_features.append(feats)
        train_labels.append(label_mapping[row['label']])
    else:
        train_failed.append(row['id'])
    if (idx+1) % 1000 == 0:
        print(f"Processed {idx+1}/{len(train_df)}")

print(f"Training: {len(train_features)} successful, {len(train_failed)} failed")
X_train = np.array(train_features)
y_train = np.array(train_labels)
print(f"X_train shape: {X_train.shape}")

# Validation
val_features = []
val_labels = []
val_failed = []

print("\nExtracting validation features (HOG+LBP)...")
for idx, row in val_df.iterrows():
    img_path = os.path.join(img_dir, row['id'])
    feats = extract_enhanced_features(img_path)
    if feats is not None:
        val_features.append(feats)
        val_labels.append(label_mapping[row['label']])
    else:
        val_failed.append(row['id'])
    if (idx+1) % 1000 == 0:
        print(f"Processed {idx+1}/{len(val_df)}")

print(f"Validation: {len(val_features)} successful, {len(val_failed)} failed")
X_val = np.array(val_features)
y_val = np.array(val_labels)
print(f"X_val shape: {X_val.shape}")

if len(train_features) == 0:
    raise ValueError("No features extracted. Check image paths.")

Extracting training features (HOG+LBP)...
Processed 1000/24402
Processed 2000/24402
Processed 3000/24402
Processed 4000/24402
Processed 5000/24402
Processed 6000/24402
Processed 7000/24402
Processed 8000/24402
Processed 9000/24402
Processed 10000/24402
Processed 11000/24402
Processed 12000/24402
Processed 13000/24402
Processed 14000/24402
Processed 15000/24402
Processed 16000/24402
Processed 17000/24402
Processed 18000/24402
Processed 19000/24402
Processed 20000/24402
Processed 21000/24402
Processed 22000/24402
Processed 23000/24402
Processed 24000/24402
Training: 24402 successful, 0 failed
X_train shape: (24402, 990)

Extracting validation features (HOG+LBP)...
Processed 1000/4307
Processed 2000/4307
Processed 3000/4307
Processed 4000/4307
Validation: 4307 successful, 0 failed
X_val shape: (4307, 990)


In [13]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Replace inf with NaN (so we can impute)
X_train_clean = np.where(np.isinf(X_train), np.nan, X_train)

# 2. Impute NaN with column median (ignore infs already turned to NaN)
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train_clean)

# 3. Now scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)

# For validation/test sets, use the same imputer and scaler
X_val_clean = np.where(np.isinf(X_val), np.nan, X_val)
X_val_imputed = imputer.transform(X_val_clean)
X_val_scaled = scaler.transform(X_val_imputed)

In [12]:


print(f"Scaled shapes: Train {X_train_scaled.shape}, Val {X_val_scaled.shape}")

# Optional: Reduce dimensionality with PCA to speed up training
# Uncomment if you want to reduce to 500 components
# from sklearn.decomposition import PCA
# pca = PCA(n_components=500, random_state=37)
# X_train_pca = pca.fit_transform(X_train_scaled)
# X_val_pca = pca.transform(X_val_scaled)
# print(f"After PCA: Train {X_train_pca.shape}, Val {X_val_pca.shape}")
# X_train_scaled, X_val_scaled = X_train_pca, X_val_pca

ValueError: Input X contains infinity or a value too large for dtype('float64').

In [14]:
import numpy as np

# 1. Check for infinite values
print("Any inf in X_train?", np.isinf(X_train).any())

# 2. Check for NaN values (StandardScaler also fails on NaN)
print("Any NaN in X_train?", np.isnan(X_train).any())

# 3. Find the maximum and minimum values
print("Max value:", np.max(X_train))
print("Min value:", np.min(X_train))

# 4. Count how many infinities per feature
inf_count_per_feature = np.sum(np.isinf(X_train), axis=0)
print("Inf count per feature:", inf_count_per_feature)

# 5. Look at the distribution of extreme values (e.g., above 1e10)
huge_values = np.abs(X_train) > 1e10
print("Number of values > 1e10:", np.sum(huge_values))

Any inf in X_train? True
Any NaN in X_train? False
Max value: inf
Min value: 0.0
Inf count per feature: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [15]:
class_counts = np.bincount(y_train)
total_samples = len(y_train)
n_classes = len(class_counts)

class_weights = total_samples / (n_classes * class_counts)
class_weights = class_weights / class_weights.sum() * n_classes

print("Class weights:")
for i in range(n_classes):
    print(f"  {inverse_mapping[i]}: {class_weights[i]:.3f}")

sample_weights = np.array([class_weights[y] for y in y_train])

Class weights:
  angry: 0.480
  disgust: 4.396
  fear: 0.468
  happy: 0.266
  neutral: 0.387
  sad: 0.397
  surprise: 0.605


In [16]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,               # reduced from 8 to avoid overfitting
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,      # lower for high-dim features
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.5,             # L1 regularization (good for many features)
    reg_lambda=1.5,            # L2 regularization
    objective='multi:softprob',
    num_class=n_classes,
    random_state=37,
    n_jobs=-1,
    early_stopping_rounds=50,
    eval_metric='mlogloss'
)

print("Training XGBoost with HOG+LBP features...")
xgb_model.fit(
    X_train_scaled,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_scaled, y_val)],
    verbose=True
)

print("Training complete!")

Training XGBoost with HOG+LBP features...
[0]	validation_0-mlogloss:1.93082
[1]	validation_0-mlogloss:1.91730
[2]	validation_0-mlogloss:1.90403
[3]	validation_0-mlogloss:1.89082
[4]	validation_0-mlogloss:1.87807
[5]	validation_0-mlogloss:1.86645
[6]	validation_0-mlogloss:1.85492
[7]	validation_0-mlogloss:1.84315
[8]	validation_0-mlogloss:1.83256
[9]	validation_0-mlogloss:1.82231
[10]	validation_0-mlogloss:1.81269
[11]	validation_0-mlogloss:1.80321
[12]	validation_0-mlogloss:1.79392
[13]	validation_0-mlogloss:1.78450
[14]	validation_0-mlogloss:1.77549
[15]	validation_0-mlogloss:1.76665
[16]	validation_0-mlogloss:1.75794
[17]	validation_0-mlogloss:1.74974
[18]	validation_0-mlogloss:1.74219
[19]	validation_0-mlogloss:1.73488
[20]	validation_0-mlogloss:1.72761
[21]	validation_0-mlogloss:1.72021
[22]	validation_0-mlogloss:1.71351
[23]	validation_0-mlogloss:1.70630
[24]	validation_0-mlogloss:1.69948
[25]	validation_0-mlogloss:1.69290
[26]	validation_0-mlogloss:1.68663
[27]	validation_0-mlogl

In [ ]:
!pip install imbalanced-learn -q
from imblearn.over_sampling import SMOTE
from collections import Counter

# Apply SMOTE
print("Applying SMOTE oversampling...")
print(f"Before SMOTE: {Counter(y_train)}")

smote = SMOTE(random_state=37, k_neighbors=3)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE: {Counter(y_train_resampled)}")

# Class weights (balanced due to SMOTE, but keep slight weighting)
class_counts_resampled = np.bincount(y_train_resampled)
total_resampled = len(y_train_resampled)
class_weights_resampled = total_resampled / (n_classes * class_counts_resampled)
class_weights_resampled = class_weights_resampled / class_weights_resampled.sum() * n_classes
sample_weights_resampled = np.array([class_weights_resampled[y] for y in y_train_resampled])

print("\nClass weights after SMOTE:")
for i in range(n_classes):
    print(f"  {inverse_mapping[i]}: {class_weights_resampled[i]:.3f}")

# XGBoost with stricter regularization for high-dim features
xgb_model = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=5,                   # Reduced from 6 to prevent overfitting on 1481 feats
    learning_rate=0.03,
    subsample=0.7,                 # Lower to reduce overfitting
    colsample_bytree=0.4,          # Lower because we have more features
    min_child_weight=8,            # Higher to make splits more conservative
    gamma=0.5,
    reg_alpha=1.0,                 # Stronger L1 (pushes many features to zero)
    reg_lambda=2.5,                # Stronger L2
    objective='multi:softprob',
    num_class=n_classes,
    random_state=37,
    n_jobs=-1,
    early_stopping_rounds=100,
    eval_metric='mlogloss'
)

print(f"\n🚀 Training XGBoost with enhanced features + SMOTE...")
print(f"Training samples: {len(X_train_resampled)}")
print(f"Feature dimension: {X_train_resampled.shape[1]}")

xgb_model.fit(
    X_train_resampled,
    y_train_resampled,
    sample_weight=sample_weights_resampled,
    eval_set=[(X_val_scaled, y_val)],
    verbose=True
)

print("\n✅ Training complete!")

# Feature importance analysis
importance = xgb_model.feature_importances_
top_features_idx = np.argsort(importance)[-20:][::-1]
print("\nTop 20 most important features (indices):", top_features_idx)
print(f"Top 20 scores: {importance[top_features_idx]}")

Applying SMOTE oversampling...
Before SMOTE: Counter({np.int64(3): 6133, np.int64(4): 4220, np.int64(5): 4105, np.int64(2): 3482, np.int64(0): 3396, np.int64(6): 2695, np.int64(1): 371})
After SMOTE: Counter({np.int64(3): 6133, np.int64(2): 6133, np.int64(4): 6133, np.int64(6): 6133, np.int64(5): 6133, np.int64(0): 6133, np.int64(1): 6133})

Class weights after SMOTE:
  angry: 1.000
  disgust: 1.000
  fear: 1.000
  happy: 1.000
  neutral: 1.000
  sad: 1.000
  surprise: 1.000

🚀 Training XGBoost with enhanced features + SMOTE...
Training samples: 42931
Feature dimension: 990
[0]	validation_0-mlogloss:1.93791
[1]	validation_0-mlogloss:1.93040
[2]	validation_0-mlogloss:1.92331
[3]	validation_0-mlogloss:1.91586
[4]	validation_0-mlogloss:1.90884
[5]	validation_0-mlogloss:1.90186
[6]	validation_0-mlogloss:1.89546
[7]	validation_0-mlogloss:1.88896
[8]	validation_0-mlogloss:1.88318
[9]	validation_0-mlogloss:1.87686
[10]	validation_0-mlogloss:1.87047
[11]	validation_0-mlogloss:1.86428
[12]	vali

In [ ]:
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('Confusion Matrix - XGBoost with HOG+LBP')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print("\nConfusion Matrix (Counts):")
print("Predicted →")
print("         ", "  ".join([f"{label:>8}" for label in unique_labels]))
for i, label in enumerate(unique_labels):
    print(f"{label:>8} ", "  ".join([f"{cm[i][j]:>8}" for j in range(len(unique_labels))]))

precision, recall, f1, support = precision_recall_fscore_support(y_val, y_pred)

print("\n" + "="*80)
print("PER-CLASS METRICS")
print("="*80)
print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-"*60)
for i, label in enumerate(unique_labels):
    print(f"{label:<12} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f} {support[i]:>10}")
print("-"*60)
print(f"{'Macro Avg':<12} {np.mean(precision):>10.4f} {np.mean(recall):>10.4f} {np.mean(f1):>10.4f} {np.sum(support):>10}")
print(f"{'Weighted Avg':<12} {np.average(precision, weights=support):>10.4f} {np.average(recall, weights=support):>10.4f} {np.average(f1, weights=support):>10.4f} {np.sum(support):>10}")
print("="*80)